# Capacidad inicial y pérdida funcional · mecanismos separados
Referencia: con transportadores de 04_compare_models. No se modifican módulos/configuraciones ni se exportan CSV/PNG. Los nuevos parámetros se muestran sólo en EXECUTED.

Se conservan los siete estados X,Nx,Ntr,Tr,G,F,E del original, su reparto de nitrógeno y sus balances. La capacidad fermentativa pasa de C=1+aTr*Tr a C=B+aTr*Tr:
- Referencia: B=1; dTr=vTr-kd*Tr.
- Capacidad inicial reducida: B=1-(1-B0)*exp(-t/tau), equivalente a dB=(1-B)/tau; transportadores sin cambios. B0 y tau comunes a todos los ensayos, no onset por LAB.
- Pérdida funcional: B=exp(-delta*t); dTr=vTr-(kd+delta)*Tr. Así dC=aTr*vTr-kd*aTr*Tr-delta*C. Se degrada capacidad constitutiva y adicional. No se combina con capacidad inicial reducida.
B es una capacidad relativa adimensional; Tr conserva la definición original. No se introduce un segundo core: B tiene solución analítica. Ambas beta modificadas se usan en G,F,E y CO2. La recuperación inicial es una hipótesis fenomenológica, no una ecuación literal de David/Zenteno. Los casos nulos B0=1 y delta=0 recuperan la referencia.

**Congelado:** química y gas del ajuste guardado de 04. Escenarios ilustrativos B0=0.2, tau=8 h; delta=0.01 h^-1 (semivida69.3h), no valores de literatura.
**Recalibrado:** referencia y variantes, dos arranques y límites químicos originales; adicionales B0[0.01,1], tau[0.25,48]h, delta[1e-7,0.05]h^-1. Química depurada LAB004–009 exclusivamente; mismas cinco variables, escalas y ponderación del original. Sin CO2 de esos labs. Se reajusta gas kLa/gain en LAB019–021 con química fija. Química LAB019–021 reservada; gas LAB016–018 holdout. No es holdout conjunto LAB019–021.
Se conserva la política de 04: medianas horarias RAW, cobertura completa y error cuadrático medio por LAB (no política censurada de05/06), solubilidad provisional, volumen y pulsos originales, Cd0=.907/.730/.829g/L en LAB019/020/021 (021 provisional), cero en LAB016–018. Se conservan las restricciones de rendimiento <=0.5114, positividad y tolerancias de integración. Temperatura de logs. No se optimizan condiciones iniciales por holdout.


In [ ]:
from pathlib import Path
import sys,json,hashlib,time
import numpy as np,pandas as pd
import matplotlib.pyplot as plt
from dataclasses import replace
from scipy.integrate import solve_ivp
from scipy.optimize import least_squares
from IPython.display import display,Markdown
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src/wine_fermentation').is_dir())
sys.path.insert(0,str(ROOT/'src'));sys.dont_write_bytecode=True
assert Path(sys.prefix).name.lower()=='fermentation'
from wine_fermentation.models.with_transporters import kinetics as original
from wine_fermentation.shared.simulation import simulate as old_simulate
from wine_fermentation.shared.gas import simulate as old_gas
from wine_fermentation.shared.solubility import saturation
from wine_fermentation import model as current
from wine_fermentation.data import natural_batches
from wine_fermentation.models.historical.calibration import load
from wine_fermentation.comparison import observations,chemical_observations,CHEMICAL_COLUMNS
protected=[p for folder in ['src','configs','notebooks','results'] for p in (ROOT/folder).rglob('*') if p.is_file() and '__pycache__' not in str(p) and not p.name.startswith('07_transporter_capacity_functional_loss.')]
before={p:hashlib.sha256(p.read_bytes()).hexdigest() for p in protected}
p0=json.loads((ROOT/'configs/models/with_transporters/fit.json').read_text())['parameters']
config=json.loads((ROOT/'configs/models/with_transporters/calibration.json').read_text())
gs=json.loads((ROOT/'configs/models/with_transporters/gas_fit.json').read_text())['experiments']['train_019_021']
g0=np.array([gs['kLa'],gs['gain']])
data,training,profiles=natural_batches(ROOT);v=load(ROOT)
LABS=['LAB019','LAB020','LAB021','LAB016','LAB017','LAB018'];TRAIN=LABS[:3]
raw=pd.read_csv(ROOT/'data/processed/co2_mass_balance_lab016_021_g_l_h.csv',usecols=['lab','time_h','record_kind','remaining_volume_L','co2_raw_g_l_h'])
obs_gas=observations(raw);evaluation={};volumes={}
for lab in LABS:
 ev=raw.loc[raw.lab.eq(lab)&raw.record_kind.eq('withdrawal_after')].sort_values('time_h');et=ev.time_h.to_numpy();evv=ev.remaining_volume_L.to_numpy()
 def volume(t,et=et,evv=evv):return np.r_[2.,evv][np.searchsorted(et,np.asarray(t,float),side='right')]
 volumes[lab]=volume;b=v.batches[lab]
 if lab in TRAIN:b=replace(b,pulses={**b.pulses,'N':tuple((t,d*2/float(volume(t))) for t,d in b.pulses['N'])})
 evaluation[lab]=b
CASES=['Referencia','Capacidad inicial reducida','Pérdida funcional']
COLORS=dict(zip(CASES,['#0072B2','#D55E00','#009E73']))
frozen={c:{**p0,'B0':1.,'tau':8.,'delta':0.} for c in CASES}
frozen[CASES[1]].update(B0=.2);frozen[CASES[2]].update(delta=.01)


In [ ]:
def activity(t,p,case):
 if case==CASES[1]:return 1-(1-p['B0'])*np.exp(-np.asarray(t)/p['tau'])
 if case==CASES[2]:return np.exp(-p['delta']*np.asarray(t))
 return np.ones_like(np.asarray(t),dtype=float)
def terms(t,y,p,b,case):
 q=original.terms(t,y,p,b);tr=max(float(y[3]),0.)
 ratio=(float(activity(t,p,case))+p['aTr']*tr)/(1+p['aTr']*tr)
 q['beta_g']*=ratio;q['beta_f']*=ratio
 return q
def rhs(t,y,p,b,case):
 X,Nx,Ntr,Tr,G,F,E=np.maximum(y,0);q=terms(t,y,p,b,case);mu,bg,bf=q['mu'],q['beta_g'],q['beta_f']
 m=q['maintenance']*q['maintenance_availability'];S=q['sugar_total']
 return [(mu-q['kd'])*X,-mu*X/p['Yxn'],-q['vTr']/p['YTrN'],q['vTr']-(q['kd']+(p['delta'] if case==CASES[2] else 0))*Tr,
 -(mu/p['Yxg']+bg/p['Yeg']+m*G/S)*X,-(mu/p['Yxf']+bf/p['Yef']+m*F/S)*X,(bg+bf)*X]
def simulate(b,p,case):
 if case==CASES[0]:return old_simulate('with_transporters',b,p)
 t=np.asarray(b.time,float);y=original.initial(b,p);out={float(t[0]):y.copy()};events=dict(b.pulses.get('N',()));now=float(t[0])
 for end in sorted(set([float(t[-1])]+[float(x) for x in events if now<x<=t[-1]])):
  requested=t[(t>now)&(t<=end)]
  sol=solve_ivp(lambda ti,yi:rhs(ti,yi,p,b,case),(now,end),y,t_eval=np.unique(np.r_[requested,end]),method='LSODA',rtol=2e-6,atol=2e-8,max_step=.75)
  tol=np.full(7,1e-6);tol[[4,5]]=1e-4
  if not sol.success or not np.isfinite(sol.y).all() or np.any(sol.y.min(axis=1)<-tol):raise RuntimeError('Invalid integration')
  out.update({float(ti):yi.copy() for ti,yi in zip(sol.t,sol.y.T)});y=sol.y[:,-1].copy();now=end
  if end in events:y=original.nutrition(y,float(events[end]),p)
 values=np.vstack([out[float(ti)] for ti in t]);f=pd.DataFrame(np.maximum(values,0),index=t,columns=original.STATES);f['N']=f.Nx+f.Ntr
 return f
b=training[0];y=original.initial(b,p0)
assert np.allclose(rhs(0,y,frozen[CASES[0]],b,CASES[0]),original.rhs(0,y,p0,b))
ref=simulate(b,frozen[CASES[0]],CASES[0])
for case in CASES[1:]:
 f=simulate(b,{**frozen[case],'B0':1.,'delta':0.},case)
 assert np.max(abs(f[list(current.STATE_NAMES)].to_numpy()-ref[list(current.STATE_NAMES)].to_numpy()))<.005
print('Reference and null hypotheses verified',flush=True)


In [ ]:
support=[]
for b in training:
 for state in current.STATE_NAMES:
  z=b.observations[state];mask=np.isfinite(z)&(b.time>0);support.append((b,state,mask,z[mask],current.sigma_for_state(state,z[mask])))
NR=sum(len(z[3]) for z in support);fit_parameters={};fit_rows=[];run_rows=[]
for case in CASES:
 bounds=dict(config['bounds'])
 if case==CASES[1]:bounds.update(B0=[.01,1.],tau=[.25,48.])
 if case==CASES[2]:bounds.update(delta=[1e-7,.05])
 names=list(bounds);lo,hi=np.array(list(bounds.values())).T;template=frozen[case].copy()
 def residual(logp):
  p={**template,**dict(zip(names,np.exp(logp)))}
  try:pred={b.batch:simulate(b,p,case) for b in training}
  except (RuntimeError,ValueError,FloatingPointError):return np.full(NR,1e4)
  return np.concatenate([(pred[b.batch][state].to_numpy()[mask]-z)/sigma for b,state,mask,z,sigma in support])
 starts=[template.copy(),template.copy()]
 if case==CASES[0]:
  starts[1].update(current.normalize_parameters(json.loads((ROOT/'configs/theta/theta_full.json').read_text())));starts[1].update(config['extra_initial'])
 elif case==CASES[1]:starts[1].update(B0=.8,tau=2.)
 else:starts[1].update(delta=.002)
 results=[];begin=time.monotonic()
 for j,start in enumerate(starts):
  sol=least_squares(residual,np.log(np.clip([start[k] for k in names],lo+1e-10,hi-1e-10)),bounds=(np.log(lo),np.log(hi)),max_nfev=100,diff_step=1e-3,ftol=2e-5,xtol=2e-5,gtol=2e-5)
  results.append(sol);run_rows.append(dict(case=case,start=j,objective=float(np.mean(sol.fun**2)),success=bool(sol.success),nfev=sol.nfev,message=sol.message))
  print(case,'start',j,'objective',run_rows[-1]['objective'],'converged',sol.success,flush=True)
 best=min(results,key=lambda s:s.fun@s.fun);p={**template,**dict(zip(names,np.exp(best.x)))};fit_parameters[case]=p
 singular=np.linalg.svd(best.jac,compute_uv=False)
 fit_rows.append(dict(case=case,objective=float(np.mean(best.fun**2)),success=bool(best.success),seconds=time.monotonic()-begin,jacobian_ratio=float(singular[-1]/singular[0]),near_bounds=[k for k,val,l,h in zip(names,np.exp(best.x),lo,hi) if min(abs(np.log(val/l)),abs(np.log(h/val)))<.02]))
 assert .1<=p['Yeg']<=.5114 and .1<=p['Yef']<=.5114
 display(pd.DataFrame([dict(parameter=k,value=p[k],lower=bounds[k][0],upper=bounds[k][1]) for k in names]))
display(pd.DataFrame(run_rows));display(pd.DataFrame(fit_rows))


In [ ]:
def gas_cache(b,p,f,case):
 t=np.unique(np.r_[np.arange(0.,f.index[-1],.1),f.index[-1],f.index])
 yy=np.column_stack([np.interp(t,f.index,f[k]) for k in original.STATES]);rate=[]
 for ti,yi in zip(t,yy):
  q=terms(ti,yi,p,b,case);rate.append((q['mu']/12.72+(q['beta_g']+q['beta_f'])/1.05)*yi[0])
 return t,np.array(rate),saturation(np.interp(t,b.time,b.temperature_c),yy[:,6],yy[:,4],yy[:,5])
def gas_forward(cache,values,lab):
 t,rate,sat=cache;k,gain=values;cd=np.zeros(len(t));cd[0]={'LAB019':.907,'LAB020':.730,'LAB021':.829}.get(lab,0.)
 out=np.zeros(len(t));out[0]=k*max(cd[0]-sat[0],0)
 for i,dt in enumerate(np.diff(t),1):
  inventory=cd[i-1]+dt*rate[i-1];cd[i]=inventory if inventory<=sat[i] else (inventory+dt*k*sat[i])/(1+dt*k);out[i]=(inventory-cd[i])/dt
 assert np.max(abs(np.diff(cd)-np.diff(t)*(rate[:-1]-out[1:])))<1e-9
 return pd.DataFrame(dict(qpred=gain*out,Cd=cd,qprod=rate),index=t)
chemical={};gases={};gasfits=[];parameters={}
for stage,pp in [('Sin recalibración',frozen),('Recalibrado',fit_parameters)]:
 for case in CASES:
  p=pp[case];parameters[stage,case]=p;caches={}
  for lab in LABS:
   f=simulate(evaluation[lab],p,case);chemical[stage,case,lab]=f;caches[lab]=gas_cache(evaluation[lab],p,f,case)
  if stage=='Sin recalibración':values=g0.copy();success=None
  else:
   def objective(logv):
    parts=[]
    for lab in TRAIN:
     f=gas_forward(caches[lab],np.exp(logv),lab);o=obs_gas[lab];o=o[o.time_h.between(f.index[0],f.index[-1])]
     parts.append((np.interp(o.time_h,f.index,f.qpred)-o.co2_raw_g_l_h.to_numpy())/np.sqrt(len(o)))
    return np.concatenate(parts)
   runs=[least_squares(objective,np.log(start),bounds=(np.log([.01,.05]),np.log([20.,3.])),max_nfev=100,ftol=1e-7,xtol=1e-7,gtol=1e-7) for start in (g0,[1.,.7])]
   best=min(runs,key=lambda s:s.fun@s.fun);values=np.exp(best.x);success=bool(best.success)
  gasfits.append(dict(stage=stage,case=case,kLa=values[0],gain=values[1],success=success,near_kLa_upper=bool(values[0]>19.6)))
  for lab in LABS:gases[stage,case,lab]=gas_forward(caches[lab],values,lab)
for lab in LABS:
 f=chemical['Sin recalibración',CASES[0],lab]
 old=old_gas('with_transporters',evaluation[lab],p0,f,dict(kLa=g0[0],gain=g0[1],Cd0={'LAB019':.907,'LAB020':.730,'LAB021':.829}.get(lab,0.)),volumes[lab])
 assert np.max(abs(old.qpred-gases['Sin recalibración',CASES[0],lab].qpred))<1e-10
display(pd.DataFrame(gasfits));display(pd.DataFrame(parameters).T)


In [ ]:
VARIABLES=[('E','Etanol',1.,'g/L'),('G','Glucosa',1.,'g/L'),('F','Fructosa',1.,'g/L'),('GF','Glucosa + fructosa',1.,'g/L'),('X','Biomasa viable',1.,'g/L'),('N','YAN',1000.,'mg/L')]
def sample(lab,key):
 if lab in TRAIN:
  o=chemical_observations(v.chem,lab,key);return o.t_h.to_numpy(),o[CHEMICAL_COLUMNS[key]].to_numpy(),'Muestras experimentales'
 if key=='GF':
  o=v.density.loc[v.density.lab.eq(lab)].dropna(subset=['t_h','density_g_l']);return o.t_h.to_numpy(),2.5*(o.density_g_l.to_numpy()-1000),'G+F por densidad'
 if key=='X':
  o=v.biomass.loc[v.biomass.lab.eq(lab)].dropna(subset=['t_h','X_viable_g_l']);return o.t_h.to_numpy(),o.X_viable_g_l.to_numpy(),'Biomasa experimental'
 return np.array([]),np.array([]),'Sin muestras disponibles'
def vals(f,key):return f.G+f.F if key=='GF' else f[key]
rows=[];counts=[]
for stage in ['Sin recalibración','Recalibrado']:
 for case in CASES:
  for b in training:
   f=simulate(b,parameters[stage,case],case)
   for key,title,scale,unit in VARIABLES:
    if key=='GF':continue
    z=b.observations[key];mask=np.isfinite(z)&(b.time>0);err=(f[key].to_numpy()[mask]-z[mask])*scale
    if len(err):rows.append(dict(stage=stage,case=case,lab=b.batch,variable=title,role='Calibración química',n=len(err),RMSE=float(np.sqrt(np.mean(err**2))),unit=unit))
  for lab in LABS:
   for key,title,scale,unit in VARIABLES:
    tt,zz,label=sample(lab,key);f=chemical[stage,case,lab];mask=(tt>0)&(tt<=f.index[-1]);err=(np.interp(tt[mask],f.index,vals(f,key))-zz[mask])*scale
    if len(err):rows.append(dict(stage=stage,case=case,lab=lab,variable=title,role='Holdout químico' if lab in TRAIN else 'Validación offline LAB016–018',n=len(err),RMSE=float(np.sqrt(np.mean(err**2))),unit=unit))
   o=obs_gas[lab];f=gases[stage,case,lab];o=o[o.time_h.between(f.index[0],f.index[-1])];err=np.interp(o.time_h,f.index,f.qpred)-o.co2_raw_g_l_h
   rows.append(dict(stage=stage,case=case,lab=lab,variable='CO₂',role='Calibración CO₂' if lab in TRAIN else 'Holdout CO₂',n=len(err),RMSE=float(np.sqrt(np.mean(err**2))),unit='g/(L h)'))
metrics=pd.DataFrame(rows);display(metrics.round(5));display(metrics.groupby(['stage','case','role','variable']).RMSE.mean().unstack('case').round(5))
for lab in LABS:
 for key,title,scale,unit in VARIABLES:
  tt,zz,label=sample(lab,key);counts.append(dict(lab=lab,variable=title,n=len(tt),source=label))
display(pd.DataFrame(counts))


In [ ]:
# Two predictions per panel, one hypothesis at a time. Gas observations are lines.
for alternative in CASES[1:]:
 for key,title,scale,unit in [('CO2','CO₂ emitido',1.,'g/(L h)'),*VARIABLES]:
  fig,axes=plt.subplots(4,3,figsize=(15,15))
  for row,stage in enumerate(['Sin recalibración','Recalibrado']):
   for j,lab in enumerate(LABS):
    ax=axes[row*2+j//3,j%3]
    for case in [CASES[0],alternative]:
     f=gases[stage,case,lab] if key=='CO2' else chemical[stage,case,lab];yy=f.qpred if key=='CO2' else vals(f,key)*scale
     ax.plot(f.index,yy,color=COLORS[case],lw=1.8,label=case)
    if key=='CO2':
     o=obs_gas[lab];ax.plot(o.time_h,o.co2_raw_g_l_h,color='black',lw=1.2,label='Medianas horarias del sensor')
    else:
     tt,zz,label=sample(lab,key);initial=tt==0
     if len(tt):
      ax.scatter(tt[~initial],zz[~initial]*scale,color='black',s=20,zorder=5,label=label)
      if initial.any():ax.scatter(tt[initial],zz[initial]*scale,facecolors='white',edgecolors='black',s=35,zorder=6,label='Muestra inicial')
     else:ax.text(.98,.95,label,ha='right',va='top',transform=ax.transAxes,fontsize=8,color='.4')
    for i,(tp,dose) in enumerate(evaluation[lab].pulses['N']):ax.axvline(tp,color='.5',ls=':',lw=.8,label='Nutrición' if i==0 else None)
    ax.set_title(lab+' · '+stage+'\n'+('Calibración CO₂ / holdout químico' if lab in TRAIN else 'Holdout CO₂ / validación offline'),fontsize=10)
    ax.set_xlabel('Tiempo desde inoculación [h]');ax.set_ylabel(title+' ['+unit+']');ax.grid(alpha=.22)
  handles={}
  for ax in axes.flat:
   hh,ll=ax.get_legend_handles_labels()
   for h,l in zip(hh,ll):handles.setdefault(l,h)
  fig.suptitle(title+' · '+alternative+' frente a referencia',fontsize=15,y=.995)
  fig.legend(handles.values(),handles.keys(),loc='upper center',bbox_to_anchor=(.5,.97),ncol=3,title='Modelos y muestras · mismos datos y límites')
  fig.tight_layout(rect=(0,0,1,.91));plt.show()
for alternative in CASES[1:]:
 fig,axes=plt.subplots(2,3,figsize=(15,8))
 for ax,b in zip(axes.flat,training):
  for case in [CASES[0],alternative]:
   f=simulate(b,fit_parameters[case],case);ax.plot(f.index,f.G+f.F,color=COLORS[case],label=case)
  o=data.loc[data.batch.eq(b.batch)];col='S_GF_g_l' if 'S_GF_g_l' in o else None
  if col:
   pts=o.dropna(subset=[col]);ax.scatter(pts.time_h,pts[col],color='black',s=20,label='G+F medido')
  else:
   mask=np.isfinite(b.observations['G'])&np.isfinite(b.observations['F']);ax.scatter(b.time[mask],(b.observations['G']+b.observations['F'])[mask],color='black',s=20,label='G+F de la misma muestra')
  ax.set_title(b.batch+' · Calibración química');ax.set_xlabel('Tiempo [h]');ax.set_ylabel('G+F [g/L]');ax.grid(alpha=.2)
 hh,ll=axes.flat[0].get_legend_handles_labels();fig.suptitle('G+F · '+alternative+' · química recalibrada',y=.99);fig.legend(hh,ll,loc='upper center',bbox_to_anchor=(.5,.95),ncol=3,title='Predicciones y muestras');fig.tight_layout(rect=(0,0,1,.85));plt.show()


In [ ]:
display(Markdown('## Evaluación y límites de interpretación'))
display(metrics[metrics.variable.eq('CO₂')].groupby(['stage','role','case']).RMSE.mean().unstack('case').round(6))
for case in CASES[1:]:
 p=fit_parameters[case];print(case,{k:p[k] for k in (['B0','tau'] if case==CASES[1] else ['delta'])})
print('Convergencia química:',[(x['case'],bool(x['success'])) for x in fit_rows])
print('Ajustes locales. Límites y Jacobiano pequeño indican incertidumbre. Un menor error CO2 no basta si empeoran X, YAN o G+F. La pérdida funcional no devuelve N al YAN extracelular.')
assert all(hashlib.sha256(p.read_bytes()).hexdigest()==h for p,h in before.items())
print('Originales protegidos e intactos:',len(before),'archivos. Sin CSV/PNG/configuraciones exportados.')


## Conclusiones del ensayo ejecutado

Seis ajustes químicos (dos arranques por caso) convergieron. La referencia congelada reproduce la capa de gas original en los seis LAB con diferencia máxima menor de 1e-10. Las hipótesis nulas recuperan las ecuaciones de referencia. Se conservaron 92 archivos originales por hashes.

| Caso recalibrado | Objetivo químico ponderado LAB004–009 | RMSE medio CO₂ LAB019–021 (ajuste) | RMSE medio CO₂ LAB016–018 (holdout) |
|---|---:|---:|---:|
| Referencia | 4.46907 | 0.16309 | 0.14731 |
| Capacidad inicial reducida | 3.60061 | 0.14671 | 0.16265 |
| Pérdida funcional | 4.46852 | 0.16322 | 0.14719 |

El objetivo químico es una media de residuos normalizados al cuadrado, no un RMSE en unidades físicas; los errores CO₂ están en g/(L·h). Los estados químicos se validan en LAB019–021 sin usarlos para la calibración biológica.

**Capacidad inicial reducida:** mejor ajuste químico, pero la solución seleccionada exige B0=0.0100001 y tau=48.0 h, prácticamente los límites exploratorios. Kie0 baja a 5.0014 g/L (límite inferior), mientras betaG0=5.1537 y betaF0=7.7582 compensan la capacidad reducida. El otro arranque converge a un ajuste distinto y peor (objetivo4.45735). La mejora no identifica de forma robusta una capacidad inicial física ni un tiempo de adaptación real. No se ampliaron límites para perseguir un ajuste mejor.

En el holdout químico LAB019–021, G+F mejora en promedio (RMSE16.19281→14.47985g/L) y etanol8.75244→7.62854g/L, pero biomasa empeora0.55845→0.57314g/L y YAN39.13046→39.58502mg/L. El CO₂ de holdout empeora. G+F estimado por densidad LAB016–018 mejora, pero sólo hay nueve puntos por LAB y tres muestras de biomasa: no demuestra una mejora de todo el horizonte.

**Pérdida funcional:** delta=3.42947e-7h^-1; prácticamente se elimina el mecanismo durante estas fermentaciones. El objetivo y las predicciones apenas cambian respecto a referencia. Con estos datos y esta formulación no aparece evidencia útil a favor de una pérdida funcional apreciable. Esto no demuestra que no exista biológicamente.

**Transferencia:** los tres ajustes mantienen kLa≈20h^-1, límite superior, con ganancias0.60323/0.62476/0.60289. Ninguna hipótesis resuelve la tendencia al límite de transferencia. Una capacidad inicial pequeña no obliga al caudal inicial a cero porque se conserva la contribución de crecimiento a la producción de CO₂.

Este ensayo modifica sólo capacidad inicial/funcional sobre el modelo previo. No implementa la propuesta completa de un único YAN y transporte normalizado. Se conservan correlación de solubilidad provisional, iniciales disueltas y política de medición de04 para aislar los mecanismos. Son resultados locales con indicios de identificación débil; no justifican sustituir el modelo original.
